In [35]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [36]:
spark=SparkSession.builder.appName("caseapp").getOrCreate()

In [38]:
customers=spark.read.option("header",True).csv("data/customers.csv",inferSchema=True)
orders=spark.read.option("header",True).csv("data/orders.csv",inferSchema=True)
products=spark.read.option("header",True).csv("data/products.csv",inferSchema=True)
returns=spark.read.option("header",True).csv("data/returns.csv",inferSchema=True)
order_items=spark.read.csv("data/order_items.csv",header=True)


In [39]:
print(customers.count())
print(orders.count())
print(products.count())
print(returns.count())


100000
1000000
50000
100000


In [40]:
df_joined=order_items.join(products, on = "product_id")
sales_category = ( df_joined
    .withColumn("selling", col("quantity") * col("selling_price"))
    .groupBy("category")
    .agg(sum("selling").alias("total_sales"))
)
sales_category.show()

sales_category.write.mode("overwrite").csv("output/q2")

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|Home & Kitchen|7.581388732799902E8|
|        Sports|7.433388681300008E8|
|   Electronics|7.442665041099958E8|
|      Clothing|7.419227945699946E8|
|         Books|7.464907783499908E8|
|        Beauty|7.626693058999963E8|
|          Toys|7.446190722999846E8|
+--------------+-------------------+



In [41]:
df_join1=orders.join(order_items,on="order_id")
df_join=df_join1.join(customers,on="customer_id")
top = (
     df_join.withColumn("selling", col("quantity")*col("selling_price"))
    .groupBy("customer_id")
    .agg(sum("selling").alias("purchase_amount"))
    .orderBy(desc("purchase_amount"))
    )
top.show(10)
top.write.mode("overwrite").csv("output/q3")

+-----------+------------------+
|customer_id|   purchase_amount|
+-----------+------------------+
|      93094|         181569.68|
|      64560|169060.39999999997|
|      23289|161573.80000000005|
|      52275|153364.78999999998|
|      61218|         153067.55|
|      52034|         152680.05|
|      40442|         151037.32|
|      60528|         148691.95|
|      84830|         148363.84|
|      82593|148281.03999999998|
+-----------+------------------+
only showing top 10 rows



In [67]:
sales=orders.join(order_items,on="order_id")
monthly_sales=( sales
    .withColumn("sales", col("quantity")* col("selling_price"))
    .withColumn("year",year("order_date"))
    .withColumn("month",month("order_date"))
    .groupBy("year","month")
    .agg(sum("sales").alias("total_sales"))
    .orderBy("year","month")
)
monthly_sales.show(12)

monthly_sales.write.mode("overwrite").csv("output/q4")

+----+-----+--------------------+
|year|month|         total_sales|
+----+-----+--------------------+
|2024|    1|4.4457777576000273E8|
|2024|    2|4.1536614419999987E8|
|2024|    3| 4.436282454099996E8|
|2024|    4| 4.278209743399952E8|
|2024|    5| 4.448106189499996E8|
|2024|    6|4.3170515406000245E8|
|2024|    7| 4.436705191199991E8|
|2024|    8| 4.410951770200014E8|
|2024|    9|4.3107152607999897E8|
|2024|   10| 4.413637893100008E8|
|2024|   11|  4.33623364040001E8|
|2024|   12| 4.427129083499969E8|
+----+-----+--------------------+



In [81]:
join1 = products.join(order_items, on="product_id")
join2 = join1.join(returns, on="order_id")

sold = join1.groupBy("category").agg(sum("quantity").alias("sold"))
returned = join2.groupBy("category").agg(sum("quantity").alias("returned"))

return_per=sold.join(returned, "category") \
    .withColumn("return_percentage",
                round(col("returned") * 100 / col("sold"), 2)) 
return_per.show()

return_per.write.mode("overwrite").csv("output/q5")

+--------------+---------+--------+-----------------+
|      category|     sold|returned|return_percentage|
+--------------+---------+--------+-----------------+
|Home & Kitchen|1302863.0|130221.0|             9.99|
|        Sports|1272271.0|127587.0|            10.03|
|   Electronics|1276881.0|127917.0|            10.02|
|      Clothing|1281718.0|128215.0|             10.0|
|         Books|1281139.0|128863.0|            10.06|
|        Beauty|1292321.0|129583.0|            10.03|
|          Toys|1292083.0|130349.0|            10.09|
+--------------+---------+--------+-----------------+



In [119]:
preferred = orders.join(customers, "customer_id")
count_df = preferred.groupBy("state", "payment_mode") \
                    .agg(count("*").alias("cnt"))
max_df = count_df.groupBy("state") \
                 .agg(max("cnt").alias("cnt"))
most_preferred = count_df.join(max_df, ["state", "cnt"]) \
                         .select("state", "payment_mode")
most_preferred.show()

most_preferred.write.mode("overwrite").csv("output/q6")

+-----+----------------+
|state|    payment_mode|
+-----+----------------+
|   NC|     Net Banking|
|   IL|Cash on Delivery|
|   OH|     Net Banking|
|   TX|             UPI|
|   GA|     Net Banking|
|   WA|             UPI|
|   CA|             UPI|
|   FL|      Debit Card|
|   NY|      Debit Card|
|   MI|      Debit Card|
+-----+----------------+

